In [1]:
import sympy as sp
from sympy import sqrt, zeros, S,  Matrix, binomial, symbols
import matplotlib.pyplot as plt
from sympy.physics.quantum import TensorProduct
from sympy.physics.quantum.trace import Tr
from sympy import lambdify, simplify
from math import comb, isnan
%matplotlib inline

In [2]:
# Identity operators for ion and photon Hilbert spaces
id_ion = sp.eye(2)
id_photon = sp.eye(5)

# Basis vectors for the ion
ket_0_ion = sp.Matrix([1, 0])
ket_1_ion = sp.Matrix([0, 1])

bra_0_ion = ket_0_ion.T
bra_1_ion = ket_1_ion.T

# Basis vectors for the photon (Assume there is at most four photon at each optical mode)
ket_0_photon = sp.Matrix([1, 0, 0, 0, 0])
ket_1_photon = sp.Matrix([0, 1, 0, 0, 0])
ket_2_photon = sp.Matrix([0, 0, 1, 0, 0])
ket_3_photon = sp.Matrix([0, 0, 0, 1, 0])
ket_4_photon = sp.Matrix([0, 0, 0, 0, 1])

bra_0_photon = ket_0_photon.T
bra_1_photon = ket_1_photon.T
bra_2_photon = ket_2_photon.T
bra_3_photon = ket_3_photon.T
bra_4_photon = ket_4_photon.T

# Define Bell states for the ions
phi_plus = (1/sqrt(2)) * (TensorProduct(ket_0_ion, ket_0_ion) + TensorProduct(ket_1_ion, ket_1_ion))
psi_plus = (1/sqrt(2)) * (TensorProduct(ket_0_ion, ket_1_ion) + TensorProduct(ket_1_ion, ket_0_ion))
psi_minus = (1/sqrt(2)) * (TensorProduct(ket_0_ion, ket_1_ion) - TensorProduct(ket_1_ion, ket_0_ion))
phi_minus = (1/sqrt(2)) * (TensorProduct(ket_0_ion, ket_0_ion) - TensorProduct(ket_1_ion, ket_1_ion))

In [3]:
a_00 = symbols('a_00')
a_11 = symbols('a_11')
a_12 = symbols('a_12')
a_21 = symbols('a_21')
a_22 = symbols('a_22')
a_33 = symbols('a_33')

In [4]:
input = Matrix([[a_00, 0, 0, 0],[0, a_11, a_12, 0],[0, a_21, a_22, 0],[0, 0, 0, a_33]])

In [5]:
def normalize(dm):
    '''
    Return the normalized version of density matrix.
    '''
    return dm / Tr(dm)

In [6]:
def apply_ED_protocol(dm):

    input_dm = TensorProduct(dm, dm)

    x_gate = Matrix([[0,1], [1,0]])

    # |a, b, a, b>
    CNOT_alice = TensorProduct(ket_0_ion @ bra_0_ion, id_ion, id_ion, id_ion) + TensorProduct(ket_1_ion @ bra_1_ion, id_ion, x_gate, id_ion)
    CNOT_bob = TensorProduct(id_ion, ket_0_ion @ bra_0_ion, id_ion, id_ion) + TensorProduct(id_ion, ket_1_ion @ bra_1_ion, id_ion, x_gate)

    dm_cnot = CNOT_bob @ (CNOT_alice @ input_dm @ CNOT_alice.T) @ CNOT_bob.T
    post_select_dm = TensorProduct(id_ion, id_ion, bra_1_ion, bra_1_ion) * dm_cnot * TensorProduct(id_ion, id_ion, ket_1_ion, ket_1_ion)
    prob = Tr(post_select_dm)
    return normalize(post_select_dm)

In [16]:
sp.print_latex(apply_ED_protocol(input)[3,3])

\frac{a_{00} a_{33}}{2 a_{00} a_{33} + 2 a_{11} a_{22}}
